In [3]:
# hud_animator.py

from __future__ import annotations

from pathlib import Path
import json
import shutil
import subprocess
from dataclasses import dataclass

import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont, ImageEnhance


# =========================================================
# CONFIG
# =========================================================

OUTPUT_FORMAT = "webm"  # "webm" | "gif" | "mp4"
FPS = 24
TOTAL_FRAMES = 120

ANIMATIONS_DIR = Path("media-site/animations")

BACKGROUND_PATH = None  # например: "mockup.png" или None
ZONES_PATH = None       # например: "zones.json" или None

CANVAS_SIZE = (900, 1600)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
PURPLE = (190, 120, 255)
WHITE = (245, 250, 255)
BG_DARK = (2, 7, 13)


# =========================================================
# BASICS
# =========================================================

def normalize_output_format(fmt: str) -> str:
    fmt = fmt.lower().strip()
    if fmt not in {"webm", "gif", "mp4"}:
        raise ValueError("OUTPUT_FORMAT должен быть: webm, gif или mp4")
    return fmt


def load_font(size: int):
    candidates = [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]

    for path in candidates:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass

    return ImageFont.load_default()


def smoothstep(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3 - 2 * t)


def default_zones(kind: str = "square") -> dict[str, list[float]]:
    if kind == "wide":
        return {
            "waveform": [0.06, 0.14, 0.94, 0.86],
            "spectrum": [0.06, 0.14, 0.94, 0.86],
        }

    return {
        "star": [0.10, 0.10, 0.90, 0.90],
        "donut": [0.10, 0.10, 0.90, 0.90],
        "internal_structure": [0.08, 0.08, 0.92, 0.92],
    }


def flatten_to_black(frame: Image.Image) -> Image.Image:
    frame = frame.convert("RGBA")
    bg = Image.new("RGBA", frame.size, (*BG_DARK, 255))
    bg.alpha_composite(frame)
    return bg.convert("RGB")


# =========================================================
# CONTEXT
# =========================================================

@dataclass
class HudContext:
    bg: Image.Image
    zones: dict
    rng: np.random.Generator
    font_small: ImageFont.FreeTypeFont
    font_tiny: ImageFont.FreeTypeFont

    @property
    def W(self) -> int:
        return self.bg.size[0]

    @property
    def H(self) -> int:
        return self.bg.size[1]

    @classmethod
    def create(
        cls,
        background_path: str | Path | None = None,
        zones_path: str | Path | None = None,
        seed: int = 26051,
        brightness: float = 0.82,
        contrast: float = 1.08,
        font_small_size: int = 22,
        font_tiny_size: int = 16,
        output_format: str = OUTPUT_FORMAT,
        canvas_size: tuple[int, int] = CANVAS_SIZE,
    ) -> "HudContext":
        output_format = normalize_output_format(output_format)

        if background_path and Path(background_path).exists():
            source_bg = Image.open(background_path).convert("RGBA")
            source_bg = ImageEnhance.Brightness(source_bg).enhance(brightness)
            source_bg = ImageEnhance.Contrast(source_bg).enhance(contrast)
            size = source_bg.size
        else:
            source_bg = None
            size = canvas_size

        if output_format == "webm":
            bg = Image.new("RGBA", size, (0, 0, 0, 0))
        else:
            bg = Image.new("RGBA", size, (*BG_DARK, 255))

        if source_bg is not None and output_format != "webm":
            bg.alpha_composite(source_bg)

        if zones_path and Path(zones_path).exists():
            with open(zones_path, "r", encoding="utf-8") as f:
                zones = json.load(f)
        else:
            zones = default_zones("square")
        
        return cls(
            bg=bg,
            zones=zones,
            rng=np.random.default_rng(seed),
            font_small=load_font(font_small_size),
            font_tiny=load_font(font_tiny_size),
        )

    def box_px(self, name: str) -> list[float]:
        if name not in self.zones:
            raise KeyError(f"Zone '{name}' not found in zones")

        x0, y0, x1, y1 = self.zones[name]
        return [x0 * self.W, y0 * self.H, x1 * self.W, y1 * self.H]

    def point_px(self, name: str) -> tuple[float, float]:
        x0, y0, x1, y1 = self.box_px(name)
        return (x0 + x1) / 2, (y0 + y1) / 2

    def star_from_area(self, name: str) -> tuple[float, float, float]:
        x0, y0, x1, y1 = self.box_px(name)
        cx = (x0 + x1) / 2
        cy = (y0 + y1) / 2
        r = min(x1 - x0, y1 - y0) / 2
        return cx, cy, r


# =========================================================
# DRAW HELPERS
# =========================================================

def draw_panel_frame(img: Image.Image, ctx: HudContext, title: str = "HUD DATA"):
    d = ImageDraw.Draw(img)

    pad = int(min(ctx.W, ctx.H) * 0.035)
    x0, y0 = pad, pad
    x1, y1 = ctx.W - pad, ctx.H - pad

    cut = int(min(ctx.W, ctx.H) * 0.045)

    pts = [
        (x0 + cut, y0),
        (x1, y0),
        (x1, y1 - cut),
        (x1 - cut, y1),
        (x0, y1),
        (x0, y0 + cut),
    ]

    d.line(pts + [pts[0]], fill=(*CYAN, 145), width=2)

    inner = pad + 18
    d.rectangle(
        [inner, inner, ctx.W - inner, ctx.H - inner],
        outline=(*CYAN2, 45),
        width=1,
    )

    d.text(
        (x0 + 18, y0 + 12),
        title,
        font=ctx.font_small,
        fill=(*CYAN2, 210),
    )


def make_star_texture(
    rng: np.random.Generator,
    size: int = 720,
    color=CYAN2,
) -> Image.Image:
    tex = Image.new("RGBA", (size, size), (0, 0, 0, 0))
    d = ImageDraw.Draw(tex)

    cx = cy = size / 2
    r = size * 0.47

    for rr in np.linspace(r, 2, 80):
        a = int(8 + 105 * (1 - rr / r) ** 1.7)
        d.ellipse([cx - rr, cy - rr, cx + rr, cy + rr], fill=(*WHITE, a))

    for _ in range(420):
        a0 = rng.uniform(0, 2 * np.pi)
        rr = rng.uniform(0.05, 0.95) * r
        x = cx + np.cos(a0) * rr
        y = cy + np.sin(a0) * rr

        length = rng.uniform(18, 75)
        a1 = a0 + rng.normal(0, 0.9)

        x2 = x + np.cos(a1) * length
        y2 = y + np.sin(a1) * length

        if (x2 - cx) ** 2 + (y2 - cy) ** 2 < r ** 2:
            alpha = int(rng.uniform(25, 95))
            d.line([x, y, x2, y2], fill=(*color, alpha), width=1)

    for _ in range(42):
        a0 = rng.uniform(0, 2 * np.pi)
        rr = rng.uniform(0.05, 0.92) * r
        x = cx + np.cos(a0) * rr
        y = cy + np.sin(a0) * rr
        rad = rng.uniform(3, 10)
        d.ellipse(
            [x - rad, y - rad, x + rad, y + rad],
            fill=(*WHITE, int(rng.uniform(120, 230))),
        )

    mask = Image.new("L", (size, size), 0)
    md = ImageDraw.Draw(mask)
    md.ellipse([cx - r, cy - r, cx + r, cy + r], fill=255)

    tex.putalpha(mask)
    return tex


def draw_live_star(
    img: Image.Image,
    ctx: HudContext,
    zone: str,
    phase: float,
    star_texture: Image.Image,
    opacity: float = 0.42,
    rotation_speed: float = 0.18,
    pulse_base: float = 0.82,
    pulse_amp: float = 0.18,
    glow_blur: float = 5,
):
    cx, cy, r = ctx.star_from_area(zone)

    layer = Image.new("RGBA", (ctx.W, ctx.H), (0, 0, 0, 0))

    rot = star_texture.rotate(
        phase * 360 / (2 * np.pi) * rotation_speed,
        resample=Image.Resampling.BICUBIC,
    )

    size = int(r * 2.08)
    rot = rot.resize((size, size), Image.Resampling.LANCZOS)

    x = int(cx - size / 2)
    y = int(cy - size / 2)

    pulse = pulse_base + pulse_amp * np.sin(phase * 2.4)

    alpha = rot.getchannel("A").point(lambda p: int(p * opacity * pulse))
    rot.putalpha(alpha)

    layer.alpha_composite(rot, (x, y))
    d = ImageDraw.Draw(layer)

    sx = ctx.W / 900

    for k in range(5):
        rr = r * (0.28 + ((phase * 0.12 + k * 0.19) % 1.0))
        a = int(90 * (1 - rr / r))
        d.ellipse(
            [cx - rr, cy - rr, cx + rr, cy + rr],
            outline=(*CYAN2, max(0, a)),
            width=max(1, int(2 * sx)),
        )

    rim_a = int(115 + 80 * np.sin(phase * 5) ** 2)
    d.ellipse(
        [cx - r, cy - r, cx + r, cy + r],
        outline=(*WHITE, rim_a),
        width=max(2, int(3 * sx)),
    )

    glow = layer.filter(ImageFilter.GaussianBlur(glow_blur))
    img.alpha_composite(glow)
    img.alpha_composite(layer)


def draw_waveform(
    img: Image.Image,
    ctx: HudContext,
    zone: str,
    phase: float,
    color=CYAN2,
    mode: str = "residual",
    line_width: int | None = None,
):
    x0, y0, x1, y1 = map(int, ctx.box_px(zone))
    d = ImageDraw.Draw(img)

    w = x1 - x0
    h = y1 - y0
    lw = line_width or max(1, int(2 * ctx.W / 900))

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 145))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 90), width=1)

    for gx in range(x0 + 20, x1 - 10, max(1, int(w / 6))):
        d.line((gx, y0 + 14, gx, y1 - 14), fill=(*CYAN, 45), width=1)

    for gy in range(y0 + 18, y1 - 10, max(1, int(h / 4))):
        d.line((x0 + 12, gy, x1 - 12, gy), fill=(*CYAN, 35), width=1)

    xs = np.linspace(x0 + 16, x1 - 16, 360)

    if mode == "amp":
        yy = np.zeros_like(xs, dtype=float)

        for k in range(4):
            center = x0 + w * (0.22 + k * 0.17 + 0.01 * np.sin(phase + k))
            yy += (0.22 + 0.15 * np.sin(phase * 1.3 + k)) * np.exp(
                -0.5 * ((xs - center) / (5 + k * 1.5)) ** 2
            )

        yy += 0.05 * np.sin(xs * 0.11 + phase * 4)
        ybase = y1 - 28
        scale = h * 0.72

    else:
        yy = (
            0.35 * np.sin(xs * 0.09 + phase * 2.0)
            + 0.18 * np.sin(xs * 0.27 - phase * 1.6)
            + 0.06 * np.sin(xs * 0.41 + phase * 4.2)
        )
        ybase = y0 + h / 2
        scale = h * 0.22

    pts = [(x, ybase - v * scale) for x, v in zip(xs, yy)]

    for p1, p2 in zip(pts[:-1], pts[1:]):
        d.line([p1, p2], fill=(*color, 235), width=lw)


def draw_spectrum_peaks(
    img: Image.Image,
    ctx: HudContext,
    zone: str,
    phase: float,
    color=CYAN2,
):
    draw_waveform(img, ctx, zone, phase, color=color, mode="amp")


def draw_progress_bar(
    img: Image.Image,
    ctx: HudContext,
    zone: str,
    progress: float,
    color=GREEN,
    segments: int = 42,
):
    x0, y0, x1, y1 = map(int, ctx.box_px(zone))
    d = ImageDraw.Draw(img)

    progress = float(np.clip(progress, 0.0, 1.0))

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 160))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 110), width=1)

    for k in range(segments):
        px = x0 + 4 + k * ((x1 - x0 - 8) / segments)
        pw = ((x1 - x0 - 8) / segments) - 2
        active = k / segments < progress
        col = color if active else CYAN
        a = int(170 if active else 45)
        d.rectangle([px, y0 + 3, px + pw, y1 - 3], fill=(*col, a))


def draw_status_text(
    img: Image.Image,
    ctx: HudContext,
    zone: str,
    phase: float,
    text: str = "ACTIVE",
    color=GREEN,
    bg_alpha: int = 155,
):
    x0, y0, x1, y1 = map(int, ctx.box_px(zone))
    d = ImageDraw.Draw(img)

    pulse = 0.45 + 0.55 * np.sin(phase * 1.6) ** 2
    alpha = int(90 + 150 * pulse)

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, bg_alpha))
    d.rectangle([x0, y0, x1, y1], outline=(*color, int(90 + 80 * pulse)), width=1)

    d.text(
        (x0 + 8, y0 + max(1, (y1 - y0) * 0.18)),
        text,
        font=ctx.font_tiny,
        fill=(*color, alpha),
    )


def draw_time_text(
    img: Image.Image,
    ctx: HudContext,
    zone: str,
    i: int,
    fps: int,
    prefix: str = "T+",
    color=CYAN2,
):
    x, y = ctx.point_px(zone)
    d = ImageDraw.Draw(img)

    d.rectangle([x - 80, y - 18, x + 155, y + 24], fill=(*BG_DARK, 190))

    seconds = int((i / fps) * 12) % 86400
    hh = 12 + seconds // 3600
    mm = (seconds // 60) % 60
    ss = seconds % 60

    d.text(
        (x - 50, y - 12),
        f"{prefix} {hh:02d}:{mm:02d}:{ss:02d}",
        font=ctx.font_small,
        fill=(*color, 230),
    )


def draw_donut_chart(
    img: Image.Image,
    ctx: HudContext,
    zone: str,
    phase: float,
    values: list[float] | None = None,
    colors: list[tuple[int, int, int]] | None = None,
    label: str = "MODE",
):
    if values is None:
        values = [
            0.35 + 0.08 * np.sin(phase * 1.3),
            0.28 + 0.06 * np.sin(phase * 1.7 + 1),
            0.22 + 0.05 * np.sin(phase * 1.1 + 2),
            0.15 + 0.04 * np.sin(phase * 1.9 + 3),
        ]

    if colors is None:
        colors = [CYAN2, GREEN, YELLOW, ORANGE]

    x0, y0, x1, y1 = map(int, ctx.box_px(zone))
    d = ImageDraw.Draw(img)

    cx = (x0 + x1) / 2
    cy = (y0 + y1) / 2
    r = min(x1 - x0, y1 - y0) * 0.43
    inner = r * 0.56

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 145))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 70), width=1)

    start = phase * 28
    total = sum(values)

    for idx, val in enumerate(values):
        extent = 360 * val / total
        pulse = 0.75 + 0.25 * np.sin(phase * 2.4 + idx)
        col = colors[idx % len(colors)]

        d.arc(
            [cx - r, cy - r, cx + r, cy + r],
            start=start,
            end=start + extent * pulse,
            fill=(*col, 210),
            width=max(3, int((r - inner) * 0.55)),
        )
        start += extent

    d.ellipse([cx - inner, cy - inner, cx + inner, cy + inner], outline=(*CYAN, 70), width=1)
    d.text((x0 + 12, y1 - 34), label, font=ctx.font_small, fill=(*CYAN2, 180))


def draw_internal_structure(
    img: Image.Image,
    ctx: HudContext,
    zone: str,
    phase: float,
    bg_alpha: int = 130,
    show_labels: bool = True,
):
    x0, y0, x1, y1 = map(int, ctx.box_px(zone))
    d = ImageDraw.Draw(img)

    w = x1 - x0
    h = y1 - y0

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, bg_alpha))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 70), width=1)

    cx = x0 + w * 0.48
    cy = y0 + h * 0.54
    r = min(w, h) * 0.42
    squash = 0.74

    def ellipse_box(rr):
        return [cx - rr, cy - rr * squash, cx + rr, cy + rr * squash]

    d.ellipse(ellipse_box(r), outline=(*CYAN2, 90), width=max(1, int(r * 0.035)))

    layers = [
        (0.78, CYAN, 75, "H/He envelope"),
        (0.56, CYAN2, 105, "C/O mantle"),
        (0.33, WHITE, 135, "degenerate core"),
        (0.15, YELLOW, 170, "hot kernel"),
    ]

    for frac, col, alpha, _label in layers:
        rr = r * frac
        pulse = 0.72 + 0.28 * np.sin(phase * (1.1 + frac) + frac * 5) ** 2
        d.ellipse(
            ellipse_box(rr),
            outline=(*col, int(alpha * pulse)),
            width=max(1, int(r * 0.025)),
        )

    wedge_pts = [
        (cx, cy),
        (cx + r * 0.88, cy - r * 0.43),
        (cx + r * 0.88, cy + r * 0.38),
    ]

    d.polygon(wedge_pts, outline=(*CYAN2, 120), fill=(*CYAN, 18))

    for frac, col, alpha, _ in layers:
        rr = r * frac

        for ang in [-0.45, -0.16, 0.18, 0.42]:
            x = cx + np.cos(ang) * rr
            y = cy + np.sin(ang) * rr * squash
            d.line([cx, cy, x, y], fill=(*col, int(alpha * 0.45)), width=1)

    for k in range(5):
        pts = []
        base_rr = r * (0.24 + k * 0.105)

        for t in np.linspace(-0.95, 0.95, 90):
            x = cx + t * base_rr * 1.22
            y = cy + (
                np.sin(t * np.pi * (2.2 + k * 0.35) + phase * 2.0 + k)
                * r * 0.035
                + k * r * 0.018
            ) * squash

            if ((x - cx) / r) ** 2 + ((y - cy) / (r * squash)) ** 2 < 0.82:
                pts.append((x, y))

        for p1, p2 in zip(pts[:-1], pts[1:]):
            d.line([p1, p2], fill=(*GREEN, 70 + k * 18), width=1)

    core_r = r * (0.09 + 0.018 * np.sin(phase * 3.2) ** 2)
    d.ellipse(
        [cx - core_r, cy - core_r * squash, cx + core_r, cy + core_r * squash],
        fill=(*YELLOW, 120),
        outline=(*WHITE, 180),
        width=1,
    )

    for k in [-0.55, 0, 0.55]:
        rr_y = r * squash * (0.28 + abs(k) * 0.25)
        d.arc([cx - r, cy - rr_y, cx + r, cy + rr_y], 0, 360, fill=(*CYAN, 35), width=1)

    for k in range(4):
        angle = phase * 0.18 + k * np.pi / 4
        xA = cx + np.cos(angle) * r
        yA = cy + np.sin(angle) * r * squash
        xB = cx - np.cos(angle) * r
        yB = cy - np.sin(angle) * r * squash
        d.line([xA, yA, xB, yB], fill=(*CYAN, 25), width=1)

    if show_labels:
        d.text((x0 + 12, y0 + 12), "WHITE DWARF CUTAWAY", font=ctx.font_small, fill=(*CYAN2, 190))
        d.text((x0 + 12, y1 - 58), "g-mode pulsation", font=ctx.font_tiny, fill=(*GREEN, 150))
        d.text((x0 + 12, y1 - 34), "C/O stratified core", font=ctx.font_tiny, fill=(*CYAN, 145))


def draw_spectral_lines(
    img: Image.Image,
    ctx: HudContext,
    zone: str,
    phase: float,
):
    x0, y0, x1, y1 = map(int, ctx.box_px(zone))
    d = ImageDraw.Draw(img)

    w = x1 - x0
    h = y1 - y0

    d.rectangle([x0, y0, x1, y1], fill=(*BG_DARK, 145))
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 80), width=1)

    xs = np.linspace(x0 + 10, x1 - 10, 420)
    continuum = y0 + h * 0.35 + 3 * np.sin(xs * 0.025 + phase)

    dips = np.zeros_like(xs)
    for k, pos in enumerate([0.18, 0.34, 0.53, 0.71, 0.86]):
        center = x0 + w * (pos + 0.006 * np.sin(phase * 0.7 + k))
        dips += (h * (0.14 + 0.04 * np.sin(phase + k))) * np.exp(
            -0.5 * ((xs - center) / 4.2) ** 2
        )

    ys = continuum + dips
    pts = list(zip(xs, ys))

    for p1, p2 in zip(pts[:-1], pts[1:]):
        d.line([p1, p2], fill=(*CYAN2, 220), width=1)

    scan_x = x0 + 10 + ((phase / (2 * np.pi)) % 1.0) * (w - 20)
    d.line([scan_x, y0 + 8, scan_x, y1 - 8], fill=(*GREEN, 120), width=1)


# =========================================================
# EXPORT
# =========================================================

def export_webm(
    frames: list[Image.Image],
    out_path: str | Path,
    fps: int = 24,
    crf: int = 34,
    frame_dir: str | Path | None = None,
    keep_frames: bool = False,
):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg не найден. Установи: brew install ffmpeg")

    frame_dir = Path(frame_dir or out_path.parent / "_frames")
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        frame.convert("RGBA").save(frame_dir / f"frame_{idx:04d}.png")

    cmd = [
        "ffmpeg", "-y",
        "-framerate", str(fps),
        "-i", str(frame_dir / "frame_%04d.png"),
        "-c:v", "libvpx-vp9",
        "-b:v", "0",
        "-crf", str(crf),
        "-pix_fmt", "yuva420p",
        "-auto-alt-ref", "0",
        "-row-mt", "1",
        str(out_path),
    ]

    subprocess.run(cmd, check=True)

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path



def export_mp4(
    frames: list[Image.Image],
    out_path: str | Path,
    fps: int = 24,
    crf: int = 22,
    frame_dir: str | Path | None = None,
    keep_frames: bool = False,
):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg не найден. Установи: brew install ffmpeg")

    frame_dir = Path(frame_dir or out_path.parent / "_frames")
    frame_dir.mkdir(parents=True, exist_ok=True)

    for idx, frame in enumerate(frames):
        flatten_to_black(frame).save(frame_dir / f"frame_{idx:04d}.png")

    cmd = [
        "ffmpeg", "-y",
        "-framerate", str(fps),
        "-i", str(frame_dir / "frame_%04d.png"),
        "-c:v", "libx264",
        "-crf", str(crf),
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        str(out_path),
    ]

    subprocess.run(cmd, check=True)

    if not keep_frames:
        shutil.rmtree(frame_dir, ignore_errors=True)

    return out_path



def export_gif(
    frames: list[Image.Image],
    out_path: str | Path,
    fps: int = 24,
):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    gif_frames = [
        flatten_to_black(frame).convert("P", palette=Image.Palette.ADAPTIVE)
        for frame in frames
    ]

    gif_frames[0].save(
        out_path,
        save_all=True,
        append_images=gif_frames[1:],
        duration=int(1000 / fps),
        loop=0,
        disposal=2,
    )

    return out_path


def export_animation(
    name: str,
    frames: list[Image.Image],
    output_format: str = OUTPUT_FORMAT,
    fps: int = FPS,
    base_dir: str | Path = ANIMATIONS_DIR,
):
    output_format = normalize_output_format(output_format)

    folder = Path(base_dir) / name
    folder.mkdir(parents=True, exist_ok=True)

    out_path = folder / f"{name}.{output_format}"

    if output_format == "webm":
        return export_webm(
            frames,
            out_path,
            fps=fps,
            frame_dir=folder / "_frames",
            keep_frames=False,
        )

    if output_format == "mp4":
        return export_mp4(
            frames,
            out_path,
            fps=fps,
            frame_dir=folder / "_frames",
            keep_frames=False,
        )

    return export_gif(frames, out_path, fps=fps)


def save_animation_set(
    animations: dict[str, list[Image.Image]],
    output_format: str = OUTPUT_FORMAT,
    fps: int = FPS,
    base_dir: str | Path = ANIMATIONS_DIR,
):
    saved = {}

    for name, frames in animations.items():
        if not frames:
            raise ValueError(f"Анимация '{name}' не содержит кадров")

        saved[name] = export_animation(
            name=name,
            frames=frames,
            output_format=output_format,
            fps=fps,
            base_dir=base_dir,
        )

        print(f"[OK] {name}: {saved[name]}")

    print(
        f"\nCreated {len(saved)} animation(s) "
        f"as '{normalize_output_format(output_format)}' in '{Path(base_dir)}'"
    )

    return saved


# =========================================================
# FRAME GENERATION
# =========================================================

def generate_frames() -> dict[str, list[Image.Image]]:
    square_size = (800, 800)
    wide_size = (1200, 420)

    ctx_square = HudContext.create(
        background_path=None,
        zones_path=None,
        output_format=OUTPUT_FORMAT,
        canvas_size=square_size,
    )

    ctx_square.zones = {
        "star": [0.10, 0.10, 0.90, 0.90],
        "donut": [0.10, 0.10, 0.90, 0.90],
        "internal_structure": [0.08, 0.08, 0.92, 0.92],
    }

    ctx_wide = HudContext.create(
        background_path=None,
        zones_path=None,
        output_format=OUTPUT_FORMAT,
        canvas_size=wide_size,
    )

    ctx_wide.zones = {
        "waveform": [0.05, 0.18, 0.95, 0.82],
        "spectrum": [0.05, 0.18, 0.95, 0.82],
    }

    star_texture = make_star_texture(ctx_square.rng)

    frames_live_star = []
    frames_waveform = []
    frames_spectrum = []
    frames_donut = []
    frames_internal_structure = []

    for i in range(TOTAL_FRAMES):

        phase = 2 * np.pi * i / TOTAL_FRAMES

        # =====================================================
        # LIVE STAR
        # =====================================================

        frame = ctx_square.bg.copy()

        draw_live_star(
            img=frame,
            ctx=ctx_square,
            zone="star",
            phase=phase,
            star_texture=star_texture,
            opacity=0.55,
        )

        frames_live_star.append(frame)

        # =====================================================
        # DONUT
        # =====================================================

        frame = ctx_square.bg.copy()

        draw_donut_chart(
            img=frame,
            ctx=ctx_square,
            zone="donut",
            phase=phase,
            label="",
        )

        frames_donut.append(frame)

        # =====================================================
        # INTERNAL STRUCTURE
        # =====================================================

        frame = ctx_square.bg.copy()

        draw_internal_structure(
            img=frame,
            ctx=ctx_square,
            zone="internal_structure",
            phase=phase,
            show_labels=False,
        )

        frames_internal_structure.append(frame)

        # =====================================================
        # WAVEFORM
        # =====================================================

        frame = ctx_wide.bg.copy()

        draw_waveform(
            img=frame,
            ctx=ctx_wide,
            zone="waveform",
            phase=phase,
        )

        frames_waveform.append(frame)

        # =====================================================
        # SPECTRUM
        # =====================================================

        frame = ctx_wide.bg.copy()

        draw_spectral_lines(
            img=frame,
            ctx=ctx_wide,
            zone="spectrum",
            phase=phase,
        )

        frames_spectrum.append(frame)

    return {
        "live_star": frames_live_star,
        "internal_structure": frames_internal_structure,
        "donut": frames_donut,
        "waveform": frames_waveform,
        "spectrum": frames_spectrum,
    }

# =========================================================
# MAIN
# =========================================================

def main():
    print("[START] hud_animator.py")
    print(f"[CONFIG] OUTPUT_FORMAT = {OUTPUT_FORMAT}")
    print(f"[CONFIG] FPS = {FPS}")
    print(f"[CONFIG] TOTAL_FRAMES = {TOTAL_FRAMES}")
    print(f"[CONFIG] ANIMATIONS_DIR = {ANIMATIONS_DIR.resolve()}")

    animations = generate_frames()

    saved = save_animation_set(
        animations=animations,
        output_format=OUTPUT_FORMAT,
        fps=FPS,
        base_dir=ANIMATIONS_DIR,
    )

    print()

    for name, path in saved.items():
        print(f"[CREATED] {name}")
        print(f"          {path.resolve()}")

    print()
    print(
        f"Created {len(saved)} animation(s) "
        f"as '{OUTPUT_FORMAT}' in '{ANIMATIONS_DIR.resolve()}'"
    )


if __name__ == "__main__":
    main()

[START] hud_animator.py
[CONFIG] OUTPUT_FORMAT = webm
[CONFIG] FPS = 24
[CONFIG] TOTAL_FRAMES = 120
[CONFIG] ANIMATIONS_DIR = /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

[OK] live_star: animations/live_star/live_star.webm


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

[OK] internal_structure: animations/internal_structure/internal_structure.webm


...

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

[OK] waveform: animations/waveform/waveform.webm


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

[OK] spectrum: animations/spectrum/spectrum.webm

Created 5 animation(s) as 'webm' in 'animations'

[CREATED] live_star
          /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/live_star/live_star.webm
[CREATED] internal_structure
          /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/internal_structure/internal_structure.webm
[CREATED] donut
          /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/donut/donut.webm
[CREATED] waveform
          /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/waveform/waveform.webm
[CREATED] spectrum
          /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations/spectrum/spectrum.webm

Created 5 animation(s) as 'webm' in '/Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/Infographics/Telemetry/animations'


[out#0/webm @ 0x12b60f6b0] video:37KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 88.201948%
frame=  120 fps= 95 q=34.0 Lsize=      69KiB time=00:00:05.00 bitrate= 113.1kbits/s speed=3.94x    
